# EDA 0 — Checkpoint A: Structure, keys, relationships (A01–A08)

**Scope:** technical data foundation only, **not** experimental evaluation. **Sources:** the three immutable Turno CSVs, with definitions from the original four-page PDF (pages 2–3). All time fields are documented as UTC. No monetary amounts, cancellations, authorizations or full host activity history are supplied.

**Evidence contract:** Every A01–A08 section includes executable evidence, reconciliation/validation and a technical limitation. Numerical observations are always at their displayed grain (**hosts**, **completed projects**, or **payment attempts**). The notebook preserves raw values as text and never silently drops or repairs records. Checkpoint B–E awaits review.

**Operational note:** From the repository root run `python -m pytest -q`, then start JupyterLab, open this file and select the Python 3.13 venv kernel. **Run All** from a clean kernel. Dataset hashes printed below must match for results to be directly comparable.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display

# Root-independent execution: works from repository root or this notebook's directory.
_root_candidates = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
ROOT = next((path for path in _root_candidates if (path / "pyproject.toml").is_file() and (path / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Repository root not found: need pyproject.toml and data/raw/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.foundation_a import RAW_FILES, EXPECTED_COLUMNS, PRIMARY_KEYS, load_raw, key_audit, audited_left_join, assert_conserved

frames, sources = load_raw(ROOT)
e, a, p = frames["exposures"], frames["assignments"], frames["payments"]
print("Execution root:", ROOT)
print("Pandas:", pd.__version__, "| Files read (no modifications):", len(sources))
display(sources)

Execution root: /mnt/data/turno-eda0-checkpoint-A
Pandas: 2.2.3 | Files read (no modifications): 3


,dataset,file,bytes,sha256,columns_unexpected
0,exposures,experiment_exposures.csv,481724,9cce67cd8d90d292c0703416b3c28c69783c56d10c2f997bf614b2bc84445327,
1,assignments,assignments.csv,16411731,c66a6d5cb89b62bc3542ee9184e75c8ded41589e91dac9d726639acad0db0ff2,
2,payments,payment_logs.csv,24857672,ab6bf97f2ca216504058b5de1deb2f80641d783e8e23651d3efc2d595675879a,


## A01 — Physical inventory, primary-key distinct counts and effective grain

**Procedure:** count rows, fields and distinct primary keys for each original file; show file fingerprints. A duplicated or null source key would contradict the documented grain and must be reported, not deduplicated. **Validation V1/V3:** `distinct + redundant key occurrences + missing keys = rows` for all three tables. **Limitation:** the PDF states the expected grain, but the file contents alone cannot establish whether all eligible real-world records were exported.

In [2]:
inventory = pd.DataFrame([
    {"dataset": name, "source_file": RAW_FILES[name], "fields": len(frame.columns),
     "expected_grain": {"exposures":"host", "assignments":"completed project", "payments":"payment attempt"}[name],
     "primary_key": PRIMARY_KEYS[name], **key_audit(frame, PRIMARY_KEYS[name])}
    for name, frame in frames.items()
]).set_index("dataset")
for name, r in inventory.iterrows():
    assert_conserved(r["rows"], r["distinct_nonnull_keys"], r["extra_key_occurrences"], r["null_keys"])
    assert r["nonnull_keys"] + r["null_keys"] == r["rows"]
    assert r["fields"] == len(frames[name].columns)
display(inventory)
print("A01 independent verification — direct len():", {k: len(v) for k,v in frames.items()})

A01 independent verification — direct len(): {'exposures': 14708, 'assignments': 279847, 'payments': 306957}


,source_file,fields,expected_grain,primary_key,rows,nonnull_keys,null_keys,distinct_nonnull_keys,repeated_key_rows,extra_key_occurrences,exact_duplicate_rows
dataset,,,,,,,,,,,
exposures,experiment_exposures.csv,3,host,host_id,14708,14708,0,14708,0,0,0
assignments,assignments.csv,5,completed project,assignment_id,279847,279847,0,279847,0,0,0
payments,payment_logs.csv,8,payment attempt,payment_log_id,306957,306957,0,306957,0,0,0


**A01 status:** answerable for the *provided extract*. The printed inventory establishes effective row granularity; it does not certify completeness of the Turno system of record.

## A02 — Null keys, duplicate keys and identical records

**Procedure:** separate (a) null primary keys, (b) non-null repeated key rows, (c) redundant key occurrences, and (d) exact duplicate rows. Do not mistake repeats of `host_id` in the project/payment tables for primary-key collisions. **Validation:** independent `value_counts(dropna=False)` for every non-null primary key and direct `duplicated()` for full-row equality. **Limitation:** distinct payment log IDs do not prove that records are distinct real-world payment events.

In [3]:
keys = inventory[["primary_key", "rows", "null_keys", "distinct_nonnull_keys", "repeated_key_rows", "extra_key_occurrences", "exact_duplicate_rows"]].copy()
independent = []
for name, frame in frames.items():
    pk = PRIMARY_KEYS[name]
    counts = frame[pk].dropna().value_counts()
    repeated_rows_check = int(counts[counts.gt(1)].sum())
    redundant_check = int(counts.sub(1).clip(lower=0).sum())
    full_dup_check = int(frame.duplicated(keep="first").sum())
    assert repeated_rows_check == keys.loc[name, "repeated_key_rows"]
    assert redundant_check == keys.loc[name, "extra_key_occurrences"]
    assert full_dup_check == keys.loc[name, "exact_duplicate_rows"]
    independent.append({"dataset": name, "repeated_key_rows_by_value_counts": repeated_rows_check,
                        "extra_key_occurrences_by_value_counts": redundant_check,
                        "exact_duplicate_rows_independent": full_dup_check})
display(keys)
display(pd.DataFrame(independent).set_index("dataset"))
print("A02 primary keys fit documented uniqueness:", bool((keys.null_keys.eq(0) & keys.extra_key_occurrences.eq(0)).all()))

A02 primary keys fit documented uniqueness: True


,primary_key,rows,null_keys,distinct_nonnull_keys,repeated_key_rows,extra_key_occurrences,exact_duplicate_rows
dataset,,,,,,,
exposures,host_id,14708,0,14708,0,0,0
assignments,assignment_id,279847,0,279847,0,0,0
payments,payment_log_id,306957,0,306957,0,0,0


,repeated_key_rows_by_value_counts,extra_key_occurrences_by_value_counts,exact_duplicate_rows_independent
dataset,,,
exposures,0,0,0
assignments,0,0,0
payments,0,0,0


**A02 status:** answerable for the extract. The result cannot resolve whether records with different IDs may describe the same underlying operation.

## A03 — Relationship cardinalities, host conflicts and guarded joins

**Expected relationships:** exposures `host_id` (1) → assignments (0..N); assignments `assignment_id` (1) → payment attempts (0..N). Cross-check payments' independent `host_id`. **Validation V2:** every many-to-one join uses `validate='many_to_one'`, checks input/output row counts, matched and unmatched partitions, and separately audits inconsistent host attribution. A failed parent key check **blocks dependent joins**. **Limitation:** joins can verify observed identifiers but cannot prove the completeness of the export.

In [4]:
# Source-key guardians: never merge with ambiguous parent keys.
for name in ["exposures", "assignments", "payments"]:
    r = inventory.loc[name]
    if r["null_keys"] or r["extra_key_occurrences"]:
        raise ValueError(f"A03 BLOCK: inconsistent source primary key in {name}; stop dependent joins")

exposure_of_assignment, j_ea = audited_left_join(
    a[["assignment_id", "host_id"]], e[["host_id", "experiment_group"]], "host_id", relation="assignments→exposures")
assignment_of_payment, j_pa = audited_left_join(
    p[["payment_log_id", "assignment_id", "host_id"]],
    a[["assignment_id", "host_id"]].rename(columns={"host_id":"assignment_host_id"}),
    "assignment_id", relation="payment_logs→assignments")
exposure_of_payment, j_pe = audited_left_join(
    p[["payment_log_id", "host_id"]], e[["host_id", "experiment_group"]],
    "host_id", relation="payment_logs→exposures")
join_audit = pd.DataFrame([j_ea, j_pa, j_pe]).set_index("relation")
assert (join_audit.multiplied_rows == 0).all()
assert_conserved(len(a), j_ea["matched"], j_ea["unmatched"])
assert_conserved(len(p), j_pa["matched"], j_pa["unmatched"])
assert_conserved(len(p), j_pe["matched"], j_pe["unmatched"])

# A single project must not refer to more than one host across its payment records.
project_host_conflicts = int(a.groupby("assignment_id", dropna=False).host_id.nunique(dropna=False).gt(1).sum())
payment_project_host_conflicts = int(p.groupby("assignment_id", dropna=False).host_id.nunique(dropna=False).gt(1).sum())
# Independent checks on relationship degree (0 counts retained in A06/A07).
host_project_degrees = a.groupby("host_id").size()
project_attempt_degrees = p.groupby("assignment_id").size()
assert int(host_project_degrees.sum()) == len(a)
assert int(project_attempt_degrees.sum()) == len(p)
display(join_audit)
display(pd.DataFrame([{"diagnostic": "assignment_id with >1 host within assignments", "count":project_host_conflicts},
                      {"diagnostic":"assignment_id with >1 host within payment logs", "count":payment_project_host_conflicts},
                      {"diagnostic":"distinct hosts seen in assignments", "count":a.host_id.nunique()},
                      {"diagnostic":"distinct projects seen in payment logs", "count":p.assignment_id.nunique()}]))

,left_rows,output_rows,matched,unmatched,multiplied_rows
relation,,,,,
assignments→exposures,279847,279847,279847,0,0
payment_logs→assignments,306957,306957,306957,0,0
payment_logs→exposures,306957,306957,306957,0,0


,diagnostic,count
0,assignment_id with >1 host within assignments,0
1,assignment_id with >1 host within payment logs,0
2,distinct hosts seen in assignments,14066
3,distinct projects seen in payment logs,279608


**A03 status:** source relationship cardinalities and observed conflicts are answerable. A valid foreign key is not proof that the operational event was correctly attributed outside the extract.

## A04 — Assignment host coverage and orphan projects

**Procedure:** inspect the assignment→exposure left join and independently use set membership on `host_id`. **Validation:** matched and unmatched assignments reconcile to all original projects. **Limitation:** the exposure file contains registered exposures; it is not proven to be the universe of originally randomized hosts.

In [5]:
a04_orphans_by_join = int(exposure_of_assignment["_match"].eq("left_only").sum())
a04_orphans_by_set = int((a.host_id.isna() | ~a.host_id.isin(e.host_id.dropna())).sum())
assert a04_orphans_by_join == a04_orphans_by_set
assert_conserved(len(a), j_ea["matched"], a04_orphans_by_join)
display(pd.DataFrame([{
    "total_projects": len(a), "projects_with_recorded_exposure":j_ea["matched"],
    "projects_without_exposure_record":a04_orphans_by_join,
    "independent_anti_join_count":a04_orphans_by_set,
    "hosts_without_exposure_record":int(a.loc[~a.host_id.isin(e.host_id),"host_id"].nunique())
}]))

,total_projects,projects_with_recorded_exposure,projects_without_exposure_record,independent_anti_join_count,hosts_without_exposure_record
0,279847,279847,0,0,0


**A04 status:** answerable within the available tables. A missing exposure record, if found, would be an association problem; it cannot be automatically interpreted as a failure of randomization.

## A05 — Payment project/host integrity, orphans and row preservation

**Procedure:** reconcile payments→projects and payments→exposures, then compare payment `host_id` with the joined project's host. **Validation V2/V8:** join indicators vs. independent set membership; mismatch classification vs. total payment log count. **Limitation:** matching identifiers do not independently validate the identity of the payer.

In [6]:
a05_orphan_project = int(assignment_of_payment._match.eq("left_only").sum())
a05_orphan_host = int(exposure_of_payment._match.eq("left_only").sum())
matched_project = assignment_of_payment._match.eq("both")
a05_mismatch = int((matched_project & assignment_of_payment.host_id.ne(assignment_of_payment.assignment_host_id).fillna(False)).sum())
a05_agree = int(matched_project.sum()) - a05_mismatch
assert_conserved(len(p), a05_orphan_project, a05_mismatch, a05_agree)
assert a05_orphan_project == int((~p.assignment_id.isin(a.assignment_id)).sum())
assert a05_orphan_host == int((~p.host_id.isin(e.host_id)).sum())
assert int((matched_project & assignment_of_payment.host_id.eq(assignment_of_payment.assignment_host_id).fillna(False)).sum()) == a05_agree
display(pd.DataFrame([{
    "all_payment_attempts":len(p), "matched_project_and_host":a05_agree,
    "project_missing":a05_orphan_project,
    "project_host_disagreement":a05_mismatch,
    "host_missing_from_exposures":a05_orphan_host,
    "unmatched_assignments_set_check":int((~p.assignment_id.isin(a.assignment_id)).sum())
}]))

,all_payment_attempts,matched_project_and_host,project_missing,project_host_disagreement,host_missing_from_exposures,unmatched_assignments_set_check
0,306957,306957,0,0,0,0


**A05 status:** technical identifier reconciliation is answerable; payment validity or amount settlement is not answered by this relationship check.

## A06 — Registered hosts with zero completed projects

**Procedure:** start at every recorded host and left-join project counts. `0` means **no completed project in the supplied extract**, not no bookings or historical inactivity. **Validation:** `zero + one + multiple = all recorded hosts`, and sum of per-host project counts equals the number of linked assignments.

In [7]:
host_counts = a.groupby("host_id", dropna=False).size().rename("n_completed_projects_in_extract").reset_index()
host_coverage, j_host_count = audited_left_join(
    e[["host_id","experiment_group","first_exposed_at"]], host_counts, "host_id", relation="registered hosts→project counts")
host_coverage["n_completed_projects_in_extract"] = host_coverage.n_completed_projects_in_extract.fillna(0).astype("int64")
host_coverage["activity_bucket"] = np.select(
    [host_coverage.n_completed_projects_in_extract.eq(0),host_coverage.n_completed_projects_in_extract.eq(1)],
    ["zero", "one"], default="multiple")
a06_by_group = pd.crosstab(host_coverage.experiment_group, host_coverage.activity_bucket)
for bucket in ["zero","one","multiple"]:
    if bucket not in a06_by_group.columns: a06_by_group[bucket]=0
a06_by_group = a06_by_group[["zero","one","multiple"]]
a06_by_group["total"] = a06_by_group.sum(axis=1)
assert_conserved(len(e), *host_coverage.activity_bucket.value_counts().tolist())
assert int(host_coverage.n_completed_projects_in_extract.sum()) == j_ea["matched"]
assert a06_by_group.total.sum() == len(e)
assert int(host_coverage.activity_bucket.eq("zero").sum()) == len(e)-int(e.host_id.isin(a.host_id).sum())
display(a06_by_group)
print("A06 registered hosts with zero completed projects:",int(host_coverage.activity_bucket.eq('zero').sum()))
print("A06 independent comparison: hosts not found in assignments:", int((~e.host_id.isin(a.host_id)).sum()))

A06 registered hosts with zero completed projects: 642
A06 independent comparison: hosts not found in assignments: 642


activity_bucket,zero,one,multiple,total
experiment_group,,,,
control,338,702,6337,7377
test,304,715,6312,7331


**A06 status:** answerable for registered hosts and completed projects in the extract. The absence of a completed project is **not** evidence of churn, rejection or inactivity in all Turno products.

## A07 — Completed projects without payment logs; technical period/group patterns

**Procedure:** aggregate counts of payment attempts by project **without** turning attempts into projects. Preserve projects with zero logs. Describe `completed_at` against the PDF baseline and experiment calendar. **Validation V1/V3:** with/without logs sum to all assignments; project attempt counts sum to all *matched* payment records; zero-log count cross-checked by set difference. **Limitation:** project created earlier than the experiment may still complete during it; absence of logs cannot be equated to nonpayment.

In [8]:
project_counts = p.groupby("assignment_id", dropna=False).size().rename("n_logs").reset_index()
project_coverage, j_ac = audited_left_join(
    a[["assignment_id", "host_id", "completed_at"]], project_counts,
    "assignment_id", relation="projects→attempt counts")
project_coverage["n_logs"] = project_coverage.n_logs.fillna(0).astype("int64")
project_coverage["has_log"] = project_coverage.n_logs.gt(0)
project_coverage = project_coverage.merge(e[["host_id","experiment_group"]],on="host_id",how="left",validate="many_to_one")

# Dates are parsed only in a derived series; unparseable source values stay intact in 'a'.
completed_utc = pd.to_datetime(project_coverage.completed_at, utc=True, format="mixed", errors="coerce")
completed_valid = completed_utc.notna()
period = pd.Series("outside_documented_windows", index=project_coverage.index, dtype="string")
period.loc[~completed_valid] = "invalid_or_missing_time"
period.loc[completed_valid & completed_utc.ge(pd.Timestamp("2025-07-18",tz="UTC")) & completed_utc.lt(pd.Timestamp("2025-08-18",tz="UTC"))] = "baseline"
period.loc[completed_valid & completed_utc.ge(pd.Timestamp("2025-08-18",tz="UTC")) & completed_utc.lt(pd.Timestamp("2025-11-20",tz="UTC"))] = "experiment"
project_coverage["completion_period"] = period
missing_logs = int(project_coverage.n_logs.eq(0).sum())
with_logs = int(project_coverage.has_log.sum())
assert_conserved(len(a),missing_logs,with_logs)
assert int(project_coverage.n_logs.sum()) == j_pa["matched"]
assert missing_logs == len(set(a.assignment_id.dropna()) - set(p.assignment_id.dropna()))

a07_groups = (project_coverage.groupby(["completion_period","experiment_group"], dropna=False)
 .agg(completed_projects=("assignment_id","size"),projects_without_logs=("has_log",lambda x:int((~x).sum())),
      attempts_in_projects=("n_logs","sum")).reset_index())
a07_groups["projects_with_logs"] = a07_groups.completed_projects - a07_groups.projects_without_logs
assert int(a07_groups.completed_projects.sum()) == len(a)
assert int(a07_groups.projects_without_logs.sum()) == missing_logs
assert int(a07_groups.attempts_in_projects.sum()) == j_pa["matched"]
display(pd.DataFrame([{"all_completed_projects":len(a),"with_logs":with_logs,"without_logs":missing_logs,
                       "independent_missing_by_sets":len(set(a.assignment_id.dropna())-set(p.assignment_id.dropna())),
                       "attempts_total":len(p), "attempts_linked_to_projects":int(project_coverage.n_logs.sum()),
                       "attempts_orphaned":j_pa["unmatched"]}]))
display(a07_groups)

,all_completed_projects,with_logs,without_logs,independent_missing_by_sets,attempts_total,attempts_linked_to_projects,attempts_orphaned
0,279847,279608,239,239,306957,306957,0


,completion_period,experiment_group,completed_projects,projects_without_logs,attempts_in_projects,projects_with_logs
0,baseline,control,38731,21,42349,38710
1,baseline,test,39814,58,43525,39756
2,experiment,control,98855,35,112083,98820
3,experiment,test,102447,125,109000,102322


**A07 status:** answerable as **absence of payment logs**, not evidence of unpaid work. `completed_at` is the project-period clock, not `created_at` or the time an attempted payment failed.

## A08 — Empirical data dictionary: schema, completeness and domains vs PDF

**Procedure:** compare all observed CSV headers to the PDF's 3/5/8 field descriptions (16 fields total). Quantify nulls and distinct values on the unchanged source representation. Derive UTC parse diagnostics separately (never silently impute or overwrite bad timestamps). **Validation:** all observed columns are represented exactly once in this dictionary; `null + nonnull = table length`; expected timestamp syntax and boolean/categorical domains are diagnostic rather than corrective. **Limitation:** observed values can support *data-format* checks but cannot independently prove the true business meaning of operational events.

In [9]:
official = {
"exposures":{"host_id":"Host ID (exposure grain)","experiment_group":"control/test","first_exposed_at":"First recorded exposure timestamp (UTC)"},
"assignments":{"assignment_id":"Completed project ID","host_id":"Host who booked project","created_at":"When project was booked (UTC)","completed_at":"When cleaner completed project (UTC)","is_first_completed_project":"Host's first-ever completed marketplace project flag"},
"payments":{"payment_log_id":"Payment attempt ID","assignment_id":"Related project ID","host_id":"Charged host ID","payment_method":"card / bank_account / credits","payment_status":"succeeded / declined / failed / voided","is_manual_attempt":"Manual-trigger indicator","created_at":"Time payment record was created (UTC)","captured_at":"When money actually moved, missing if not captured (UTC)"}
}
expected_domains={
("exposures","experiment_group"):{"control","test"},
("assignments","is_first_completed_project"):{"True","False","true","false"},
("payments","payment_method"):{"card","bank_account","credits"},
("payments","payment_status"):{"succeeded","declined","failed","voided"},
("payments","is_manual_attempt"):{"True","False","true","false"},
}
timestamp_fields={"exposures":{"first_exposed_at"},"assignments":{"created_at","completed_at"},
                  "payments":{"created_at","captured_at"}}
dictionary=[]
for name, frame in frames.items():
    for col in frame.columns:
        raw=frame[col]; null_n=int(raw.isna().sum()); nonnull_n=len(frame)-null_n
        actual=set(raw.dropna().unique()) if (name,col) in expected_domains else None
        unknown=sorted(actual-expected_domains[(name,col)]) if actual is not None else []
        if col in timestamp_fields[name]:
            parsed=pd.to_datetime(raw,utc=True,format="mixed",errors="coerce")
            invalid_nonempty=int((raw.notna() & parsed.isna()).sum())
            format_label="UTC timestamp text"
        elif col.endswith("_id"):
            invalid_nonempty=None;format_label="identifier text"
        elif (name,col) in expected_domains:
            invalid_nonempty=None;format_label="categorical/boolean source text"
        else: invalid_nonempty=None;format_label="source text (unspecified)"
        dictionary.append({"dataset":name,"field":col,"PDF_definition":official.get(name,{}).get(col,"NOT DOCUMENTED"),
            "observed_read_type":str(raw.dtype),"technical_format":format_label,
            "rows":len(frame),"nonnull":nonnull_n,"null":null_n,
            "null_pct":round(100*null_n/len(frame),3) if len(frame) else None,
            "distinct_nonnull":int(raw.nunique()),
            "unexpected_domain_values":str(unknown) if actual is not None else "not applicable",
            "unparseable_nonnull_timestamps":invalid_nonempty})
dictionary_df=pd.DataFrame(dictionary)
assert len(dictionary_df) == sum(map(len,[e.columns,a.columns,p.columns]))
assert len(dictionary_df)==len(dictionary_df[["dataset","field"]].drop_duplicates())
assert (dictionary_df.nonnull + dictionary_df["null"] == dictionary_df.rows).all()
assert all(set(EXPECTED_COLUMNS[n]).issubset(set(frames[n].columns)) for n in frames)
display(dictionary_df)
print("A08 parsed timestamp failures (excluding missing fields):",
      dictionary_df.loc[dictionary_df.unparseable_nonnull_timestamps.notna(),
       ["dataset","field","unparseable_nonnull_timestamps"]].to_dict("records"))
print("A08 schema extras:", sources[["dataset","columns_unexpected"]].to_dict("records"))

A08 parsed timestamp failures (excluding missing fields): [{'dataset': 'exposures', 'field': 'first_exposed_at', 'unparseable_nonnull_timestamps': 0.0}, {'dataset': 'assignments', 'field': 'created_at', 'unparseable_nonnull_timestamps': 0.0}, {'dataset': 'assignments', 'field': 'completed_at', 'unparseable_nonnull_timestamps': 0.0}, {'dataset': 'payments', 'field': 'created_at', 'unparseable_nonnull_timestamps': 0.0}, {'dataset': 'payments', 'field': 'captured_at', 'unparseable_nonnull_timestamps': 0.0}]
A08 schema extras: [{'dataset': 'exposures', 'columns_unexpected': ''}, {'dataset': 'assignments', 'columns_unexpected': ''}, {'dataset': 'payments', 'columns_unexpected': ''}]


,dataset,field,PDF_definition,observed_read_type,technical_format,rows,nonnull,null,null_pct,distinct_nonnull,unexpected_domain_values,unparseable_nonnull_timestamps
0,exposures,host_id,Host ID (exposure grain),string,identifier text,14708,14708,0,0.000,14708,not applicable,NaN
1,exposures,experiment_group,control/test,string,categorical/boolean source text,14708,14708,0,0.000,2,[],NaN
2,exposures,first_exposed_at,First recorded exposure timestamp (UTC),string,UTC timestamp text,14708,14708,0,0.000,14609,not applicable,0.0
3,assignments,assignment_id,Completed project ID,string,identifier text,279847,279847,0,0.000,279847,not applicable,NaN
4,assignments,host_id,Host who booked project,string,identifier text,279847,279847,0,0.000,14066,not applicable,NaN
5,assignments,created_at,When project was booked (UTC),string,UTC timestamp text,279847,279847,0,0.000,276408,not applicable,0.0
6,assignments,completed_at,When cleaner completed project (UTC),string,UTC timestamp text,279847,279847,0,0.000,269387,not applicable,0.0
7,assignments,is_first_completed_project,Host's first-ever completed marketplace project flag,string,categorical/boolean source text,279847,279847,0,0.000,2,[],NaN
8,payments,payment_log_id,Payment attempt ID,string,identifier text,306957,306957,0,0.000,306957,not applicable,NaN
9,payments,assignment_id,Related project ID,string,identifier text,306957,306957,0,0.000,279608,not applicable,NaN


**A08 status:** answerable for physical schema and the PDF-defined fields. `observed_read_type=string` is intentional: read-time numeric/boolean coercion would risk concealing unexpected representations. All timestamps are only **diagnostically** parsed into new UTC series. This is not an approval of any metric-definition rule.

## Checkpoint A — Controls, dependencies and handoff

- **V1:** A01 totals, A05 payment partition, A06 host partition, A07 project partition.
- **V2:** A03 guarded many-to-one joins, with recorded join counts and explicit row-preservation checks.
- **V3:** Host aggregates reconcile to linked project counts, project attempt aggregates to linked payment logs.
- **V5:** Only derived UTC interpretation of completed timestamps for A07; comprehensive temporal observability is deferred to Checkpoint C.
- **V6/V8:** `pytest` synthetic corner cases + independent anti-join/set checks and `value_counts` verification.
- **V7:** Run All in an isolated kernel and preserve outputs for review.

**STOP:** Before implementing B09–B17, review this notebook, the technical anomaly inventory and any source/join integrity blockers. This notebook does **not** infer payment success, full financial settlement, treatment effects, or valid analysis eligibility. No records are excluded.

In [10]:
blockers = []
for name in frames:
    if inventory.loc[name,"null_keys"] or inventory.loc[name,"extra_key_occurrences"]:
        blockers.append(f"{name}: invalid primary key")
if any(join_audit.multiplied_rows):blockers.append("unexplained join multiplication")
if a05_orphan_project or a05_orphan_host or a05_mismatch:blockers.append("orphaned or mismatched payment identifiers")
print("CHECKPOINT A COMPLETE | A01–A08: 8 evidence sections")
print("Technical relationship blockers:",blockers if blockers else "none detected in supplied CSVs")
print("Review flags (not automatically blockers):",{"projects_without_logs":missing_logs,
                                                  "registered_hosts_without_completed_projects":int(host_coverage.activity_bucket.eq('zero').sum()),
                                                  "noncapture_missing_timestamps":int(p.captured_at.isna().sum())})

CHECKPOINT A COMPLETE | A01–A08: 8 evidence sections
Technical relationship blockers: none detected in supplied CSVs
Review flags (not automatically blockers): {'projects_without_logs': 239, 'registered_hosts_without_completed_projects': 642, 'noncapture_missing_timestamps': 27672}
